In [ ]:
# %pip install bt pandas numpy matplotlib polars mlforecast

In [179]:
# %pip install hyperopt

In [3]:
# %pip install numpy==1.25.2 pandas==2.0.3 gym_mtsim

In [4]:
import numpy as np
print(np.__version__)
import pandas as pd
print(pd.__version__)

1.25.2
2.0.3


In [ ]:
import pandas as pd
import numpy as np
import polars as pl
import pickle
import sys
sys.path.append("C:/Users/WilliamFetzner/Documents/Trading/")
from gym_mtsim_forked.gym_mtsim.data import FOREX_DATA_PATH, FOREX_DATA_PATH_15MIN
import pytz
from datetime import datetime, timedelta, time
from gym_mtsim import MtSimulator, OrderType
from hyperopt import hp, fmin, tpe, STATUS_OK, Trials, STATUS_FAIL

In [6]:
path_to_trading_folder = 'C:/Users/WilliamFetzner/Documents/Trading/'
# data_path = 'EURUSD_full_tickstory_data_15_min.csv'
data_path = 'EURUSD_full_tickstory_data_hourly.csv'

In [ ]:
with open(FOREX_DATA_PATH_15MIN, 'rb') as f:
    symbols_1hr = pickle.load(f)
# convert symbols_1hr to a pd.dataframe
symbols_1hr[1]['EURUSD'].index = pd.to_datetime(symbols_1hr[1]['EURUSD'].index)
full_data = symbols_1hr[1]['EURUSD']
# symbols_1hr[1]['EURUSD'] = full_15min
# with open(FOREX_DATA_PATH_15MIN, 'wb') as f:
#     pickle.dump(symbols_1hr, f)
full_data = full_data[full_data.index.year >= 2024]
full_data

,Open,High,Low,Close
Time,,,,
2016-08-01 03:00:00+00:00,1.11672,1.11767,1.11660,1.11750
2016-08-01 04:00:00+00:00,1.11750,1.11782,1.11683,1.11719
2016-08-01 05:00:00+00:00,1.11719,1.11753,1.11698,1.11709
2016-08-01 06:00:00+00:00,1.11710,1.11785,1.11662,1.11785
2016-08-01 07:00:00+00:00,1.11784,1.11814,1.11752,1.11794
...,...,...,...,...
2024-08-09 19:00:00+00:00,1.09240,1.09245,1.09178,1.09186
2024-08-09 20:00:00+00:00,1.09186,1.09234,1.09151,1.09206
2024-08-09 21:00:00+00:00,1.09207,1.09217,1.09170,1.09195


In [ ]:
from typing import Tuple, Dict

class SDMomentumStrategy:
    def __init__(
        self,
        # ADX variables     
        adx_smoothing = 14,
        adx_di_length = 14,

        # SuperTrend variables
        supertrend_atr_length = 10,
        supertrend_factor = 3,
        
        # impulse variables
        lengthMA = 34,
        lengthSignal = 9,

        # Bollinger Bands variables
        bb_period = 20,
        bb_std = 2,
    ):
        self.lengthMA = lengthMA
        self.lengthSignal = lengthSignal
        self.adx_smoothing = adx_smoothing
        self.adx_di_length = adx_di_length
        self.supertrend_atr_length = supertrend_atr_length
        self.supertrend_factor = supertrend_factor
        self.bb_period = bb_period
        self.bb_std = bb_std

    def calculate_bollinger_bands(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculate Bollinger Bands"""
        df['bb_middle'] = df['close'].rolling(window=self.bb_period).mean()
        std = df['close'].rolling(window=self.bb_period).std()
        df['bb_upper'] = df['bb_middle'] + (std * self.bb_std)
        df['bb_lower'] = df['bb_middle'] - (std * self.bb_std)
        df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['bb_middle']
        return df

    def calculate_keltner_channels(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculate Keltner Channels"""
        typical_price = (df['high'] + df['low'] + df['close']) / 3
        df['kc_middle'] = typical_price.rolling(window=self.keltner_period).mean()
        
        # Calculate ATR
        high_low = df['high'] - df['low']
        high_close = np.abs(df['high'] - df['close'].shift())
        low_close = np.abs(df['low'] - df['close'].shift())
        ranges = pd.concat([high_low, high_close, low_close], axis=1)
        true_range = ranges.max(axis=1)
        atr = true_range.rolling(window=self.keltner_period).mean()
        
        df['kc_upper'] = df['kc_middle'] + (atr * self.keltner_atr_multiplier)
        df['kc_lower'] = df['kc_middle'] - (atr * self.keltner_atr_multiplier)
        return df

    def calculate_rsi(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculate RSI"""
        delta = df['close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=self.rsi_period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=self.rsi_period).mean()
        rs = gain / loss
        df['rsi'] = 100 - (100 / (1 + rs))
        return df

    def calculate_stochastic(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculate Stochastic Oscillator"""
        low_min = df['low'].rolling(window=self.stoch_period).min()
        high_max = df['high'].rolling(window=self.stoch_period).max()
        
        k_fast = 100 * (df['close'] - low_min) / (high_max - low_min)
        df['stoch_k'] = k_fast.rolling(window=self.stoch_smoothk).mean()
        df['stoch_d'] = df['stoch_k'].rolling(window=self.stoch_smoothd).mean()
        return df

    def calculate_roc(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculate Rate of Change"""
        df['roc'] = ((df['close'] - df['close'].shift(self.roc_period)) / 
                     df['close'].shift(self.roc_period)) * 100
        return df

    def calculate_volatility_state(self, df: pd.DataFrame) -> pd.DataFrame:
        """Determine if we're in high or low volatility state"""
        df['volatility'] = df['close'].rolling(window=20).std()
        df['vol_ma'] = df['volatility'].rolling(window=self.vol_lookback).mean()
        df['high_volatility'] = df['volatility'] > df['vol_ma']
        return df

    def check_squeeze(self, df: pd.DataFrame) -> pd.DataFrame:
        """Check if price is in a squeeze (Bollinger Bands inside Keltner Channels)"""
        df['squeeze'] = (df['bb_upper'] <= df['kc_upper']) & (df['bb_lower'] >= df['kc_lower'])
        return df

    def generate_signals(self, df: pd.DataFrame) -> pd.DataFrame:
        """Generate trading signals based on strategy rules"""
        # Initialize signal column
        df['signal'] = 0
        
        # Long signal conditions
        long_conditions = (
            (df['squeeze'].shift(1) & ~df['squeeze']) &  # Squeeze just ended
            (df['rsi'] > 40) &  # RSI confirms
            (df['stoch_k'] > 20) &  # Stochastic confirms
            (df['roc'] > 0)  # ROC confirms
        )
        
        # Short signal conditions
        short_conditions = (
            (df['squeeze'].shift(1) & ~df['squeeze']) &  # Squeeze just ended
            (df['rsi'] < 60) &  # RSI confirms
            (df['stoch_k'] < 80) &  # Stochastic confirms
            (df['roc'] < 0)  # ROC confirms
        )
        
        # Set signals
        df.loc[long_conditions, 'signal'] = 1
        df.loc[short_conditions, 'signal'] = -1
        
        return df

    def calculate_position_size(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculate position size based on ATR"""
        # Calculate ATR-based position sizing
        df['atr'] = df['high'].rolling(window=14).max() - df['low'].rolling(window=14).min()
        df['position_size'] = 1.0  # Base position size
        
        # Adjust for volatility
        df.loc[df['high_volatility'], 'position_size'] *= 0.5
        
        return df

    def apply_strategy(self, df: pd.DataFrame) -> pd.DataFrame:
        """Apply the complete strategy to the dataframe"""
        # Calculate all indicators
        df = self.calculate_bollinger_bands(df)
        df = self.calculate_keltner_channels(df)
        df = self.calculate_rsi(df)
        df = self.calculate_stochastic(df)
        df = self.calculate_roc(df)
        df = self.calculate_volatility_state(df)
        df = self.check_squeeze(df)
        
        # Generate signals and position sizes
        df = self.generate_signals(df)
        df = self.calculate_position_size(df)
        
        # Calculate stop levels
        df['stop_distance'] = df['atr'] * 1.5  # Use 1.5 * ATR for stop distance
        df['stop_level'] = np.where(
            df['signal'] == 1,
            df['close'] - df['stop_distance'],
            np.where(
                df['signal'] == -1,
                df['close'] + df['stop_distance'],
                np.nan
            )
        )
        
        return df

    def get_trade_info(self, df: pd.DataFrame, index: int) -> Dict:
        """Get trade information for a specific signal"""
        row = df.iloc[index]
        return {
            'signal': row['signal'],
            'entry_price': row['close'],
            'stop_level': row['stop_level'],
            'position_size': row['position_size'],
            'volatility_state': 'high' if row['high_volatility'] else 'low',
            'squeeze_state': row['squeeze'],
            'rsi': row['rsi'],
            'stoch_k': row['stoch_k'],
            'roc': row['roc']
        }


In [154]:
def find_seconds_to_next_news(df, news_counts):
    # Ensure both DataFrames have their time columns as datetime
    df['Time'] = pd.to_datetime(df['Time'])
    news_counts['datetime'] = pd.to_datetime(news_counts['datetime'])

    # Combine both DataFrames
    combined = pd.concat([
        df[['Time']].assign(source='main').rename(columns={'Time': 'time'}),
        news_counts[['datetime']].assign(source='news').rename(columns={'datetime': 'time'})
    ])

    # Sort by time and source
    combined = combined.sort_values(by=['time', 'source'], ascending=[True, False])

    # Find the next and previous news events
    result = combined.copy()
    result['prev_news_time'] = np.where(result['source'] == 'news', result['time'], None)
    result['next_news_time'] = np.where(result['source'] == 'news', result['time'], None)
    
    result['prev_news_time'] = result['prev_news_time'].ffill()
    result['next_news_time'] = result['next_news_time'].bfill()

    # Filter for main events and calculate time differences
    result = result[result['source'] == 'main'].copy()
    
    result['time_week_nbr'] = result['time'].dt.isocalendar().week
    result['prev_time_week_nbr'] = result['prev_news_time'].dt.isocalendar().week
    result['next_time_week_nbr'] = result['next_news_time'].dt.isocalendar().week

    # Calculate seconds since last news event
    result['seconds_since_last_news_event'] = np.where(
        result['time'].dt.isocalendar().week != result['prev_news_time'].dt.isocalendar().week,
        (result['prev_news_time'] - result['time']).dt.total_seconds() + (2 * 86400),
        (result['prev_news_time'] - result['time']).dt.total_seconds()
    )

    # Calculate seconds to next news event
    result['seconds_to_next_news_event'] = np.where(
        result['time'].dt.isocalendar().week != result['next_news_time'].dt.isocalendar().week,
        (result['next_news_time'] - result['time']).dt.total_seconds() - (2 * 86400),
        (result['next_news_time'] - result['time']).dt.total_seconds()
    )

    # Join back to original DataFrame
    final_df = df.merge(
        result[['time', 'seconds_since_last_news_event', 'seconds_to_next_news_event']],
        left_on='Time',
        right_on='time',
        how='left'
    ).drop(columns=['time'])

    return final_df


In [168]:
news_events_gr = pd.read_csv(
    f"{path_to_trading_folder}data_files/calendar_df_full_updated.csv", 
    parse_dates=['datetime']
).groupby('datetime').count()
news_events_gr.loc[:, 'event'] = (news_events_gr['Id'] >= 1).astype(int)
news_events_gr_id_drp = news_events_gr.drop(columns='Id')
# add timezone "UTC" but don't change the times
news_events_gr_id_drp.index = news_events_gr_id_drp.index.tz_localize('UTC', ambiguous='infer')
news_events_gr_id_drp = news_events_gr_id_drp.reset_index()
news_events_gr_id_drp.tail()

,datetime,event
3176,2024-08-05 17:00:00+00:00,1
3177,2024-08-06 12:00:00+00:00,1
3178,2024-08-13 15:30:00+00:00,1
3179,2024-08-14 12:00:00+00:00,1
3180,2024-08-14 15:30:00+00:00,1


In [ ]:
from turtle import end_fill


strategy = SDMomentumStrategy(
    bb_period=int(params['bb_period']),
    bb_std=params['bb_std'],
    keltner_period=int(params['keltner_period']),
    keltner_atr_multiplier=params['keltner_atr_multiplier'],
    rsi_period=int(params['rsi_period']),
    stoch_period=int(params['stoch_period']),
    roc_period=int(params['roc_period'])
)

full_data.columns = full_data.columns.str.lower()
full_data_index_reset = full_data.reset_index()
full_data_w_strategy = strategy.apply_strategy(full_data_index_reset)
full_data_w_strategy_news = find_seconds_to_next_news(full_data_w_strategy, 
                                                        news_events_gr_id_drp)
full_data_w_strategy_news.loc[:, 'Week_Nbr'] = (
    full_data_w_strategy_news['Time'].dt.year.astype(str) + 
    full_data_w_strategy_news['Time'].dt.isocalendar().week.astype(str)
)
total_wks = len(full_data_w_strategy_news.Week_Nbr.unique())

sim = MtSimulator(
    unit='USD',
    balance=400_000.,
    leverage=100.,
    stop_out_level=0.2,
    hedge=True,
    symbols_filename=FOREX_DATA_PATH
)
open_orders = {}
start_hour = 10
end_hour = 23
for _, row in full_data_w_strategy_news.iterrows():    
    # check if any of the stop_losses have been hit, or if any of the exit
    # conditions have been met for any of the open trades if not have the 
    # stoploss trail behind the others
    sim.current_time = row['Time']
    long_exit_boolean = (
        (row['close'] >= row['bb_upper']) |  # Price hits upper Bollinger Band
        (row['rsi'] > 70) |  # Overbought on RSI
        (row['stoch_k'] > 80) |  # Overbought on Stochastic
        (row['roc'] < -2) |  # Momentum reversal
        (row['close'] < row['kc_middle']) | # Price below Keltner middle line
        (row['Time'].time() >= time(23, 0, 0)) # Negative Swap protection
    )
    short_exit_boolean = (
        (row['close'] <= row['bb_lower']) |  # Price hits lower Bollinger Band
        (row['rsi'] < 30) |  # Oversold on RSI
        (row['stoch_k'] < 20) |  # Oversold on Stochastic
        (row['roc'] > 2) |  # Momentum reversal
        (row['close'] > row['kc_middle']) | # Price above Keltner middle line
        ((row['Time'].weekday() == 4) & 
        (row['Time'].time() >= time(23, 0, 0))) # Weekend hold protection
    )
    orders_to_remove = []
    for order in open_orders:
        # if the exit conditions have been met, or the stoploss has been hit
        # or the opposite signal has been hit, close the order
        if order.type == OrderType.Buy:
            if ((long_exit_boolean) or (open_orders[order] >= row['low']) or 
            (row['signal'] < 0)): 
                sim.close_order(order)
                orders_to_remove.append(order)
            # otherwise update the trailing stoploss
            else:
                open_orders[order] = max(
                    open_orders[order], row['close'] + row['stop_distance'])
                
        elif order.type == OrderType.Sell:
            if ((short_exit_boolean) or (open_orders[order] <= row['high'])
                or (row['signal'] > 0)):
                sim.close_order(order)
                orders_to_remove.append(order)
            else:
                open_orders[order] = min(
                    open_orders[order], row['close'] - row['stop_distance'])
    for o in orders_to_remove:
        open_orders.pop(o)

    if ((row['signal'] > 0) and (row['seconds_to_next_news_event'] > 900) and 
        (row['seconds_since_last_news_event'] < -900) and 
        ((row['Time'].time() >= time(start_hour, 0, 0)) and (row['Time'].time() <= time(end_hour, 0, 0))) and
        (row['Time'].time() < time(23, 0, 0))):
        long_order = sim.create_order(
            order_type=OrderType.Buy,
            symbol='EURUSD',
            volume=row['position_size'],
            fee=max(0., np.random.normal(0.0001, 0.00003)),
        )
        long_stop_loss = long_order.entry_price - row['stop_distance']
        open_orders[long_order] = long_stop_loss


    if ((row['signal'] < 0) and (row['seconds_to_next_news_event'] > 900) and 
        (row['seconds_since_last_news_event'] < -900) and 
        ((row['Time'].time() >= time(start_hour, 0, 0)) and (row['Time'].time() <= time(end_hour, 0, 0))) and
        (((row['Time'].time() < time(23, 0, 0)) and (row['Time'].weekday() == 4)) or 
        row['Time'].weekday() != 4)): 
        short_order = sim.create_order(
            order_type=OrderType.Sell,
            symbol='EURUSD',
            volume=row['position_size'],
            fee=max(0., np.random.normal(0.0001, 0.00003)),
        )
        short_stop_loss = short_order.entry_price + row['stop_distance']
        open_orders[short_order] = short_stop_loss
state = sim.get_state()
if len(state['orders']) > 0:
    reward = state['orders']['Profit'].sum()
else:
    reward = 0

print(
    f"balance: {state['balance']}, profit: {reward}, total orders: {len(state['orders'])}"
)

reward *= -1 
if (len(state['orders']) < total_wks):
    reward += float('inf')

In [203]:
def objective(params):
    strategy = SDMomentumStrategy(
        bb_period=int(params['bb_period']),
        bb_std=params['bb_std'],
        keltner_period=int(params['keltner_period']),
        keltner_atr_multiplier=params['keltner_atr_multiplier'],
        rsi_period=int(params['rsi_period']),
        stoch_period=int(params['stoch_period']),
        roc_period=int(params['roc_period'])
    )

    full_data.columns = full_data.columns.str.lower()
    full_data_index_reset = full_data.reset_index()
    full_data_w_strategy = strategy.apply_strategy(full_data_index_reset)
    full_data_w_strategy_news = find_seconds_to_next_news(full_data_w_strategy, 
                                                          news_events_gr_id_drp)
    full_data_w_strategy_news.loc[:, 'Week_Nbr'] = (
        full_data_w_strategy_news['Time'].dt.year.astype(str) + 
        full_data_w_strategy_news['Time'].dt.isocalendar().week.astype(str)
    )
    total_wks = len(full_data_w_strategy_news.Week_Nbr.unique())

    sim = MtSimulator(
        unit='USD',
        balance=400_000.,
        leverage=100.,
        stop_out_level=0.2,
        hedge=True,
        symbols_filename=FOREX_DATA_PATH
    )
    open_orders = {}
    for _, row in full_data_w_strategy_news.iterrows():    
        # check if any of the stop_losses have been hit, or if any of the exit
        # conditions have been met for any of the open trades if not have the 
        # stoploss trail behind the others
        sim.current_time = row['Time']
        long_exit_boolean = (
            (row['close'] >= row['bb_upper']) |  # Price hits upper Bollinger Band
            (row['rsi'] > 70) |  # Overbought on RSI
            (row['stoch_k'] > 80) |  # Overbought on Stochastic
            (row['roc'] < -2) |  # Momentum reversal
            (row['close'] < row['kc_middle']) | # Price below Keltner middle line
            (row['Time'].time() >= time(23, 0, 0)) # Negative Swap protection
        )
        short_exit_boolean = (
            (row['close'] <= row['bb_lower']) |  # Price hits lower Bollinger Band
            (row['rsi'] < 30) |  # Oversold on RSI
            (row['stoch_k'] < 20) |  # Oversold on Stochastic
            (row['roc'] > 2) |  # Momentum reversal
            (row['close'] > row['kc_middle']) | # Price above Keltner middle line
            ((row['Time'].weekday() == 4) & 
            (row['Time'].time() >= time(23, 0, 0))) # Weekend hold protection
        )
        orders_to_remove = []
        for order in open_orders:
            # if the exit conditions have been met, or the stoploss has been hit
            # or the opposite signal has been hit, close the order
            if order.type == OrderType.Buy:
                if ((long_exit_boolean) or (open_orders[order] >= row['low']) or 
                (row['signal'] < 0)): 
                    sim.close_order(order)
                    orders_to_remove.append(order)
                # otherwise update the trailing stoploss
                else:
                    open_orders[order] = max(
                        open_orders[order], row['close'] + row['stop_distance'])
                    
            elif order.type == OrderType.Sell:
                if ((short_exit_boolean) or (open_orders[order] <= row['high'])
                    or (row['signal'] > 0)):
                    sim.close_order(order)
                    orders_to_remove.append(order)
                else:
                    open_orders[order] = min(
                        open_orders[order], row['close'] - row['stop_distance'])
        for o in orders_to_remove:
            open_orders.pop(o)

        if ((row['signal'] > 0) and (row['seconds_to_next_news_event'] > 900) and 
            (row['seconds_since_last_news_event'] < -900) and 
            ((row['Time'].time() >= time(10, 0, 0)) and (row['Time'].time() <= time(17, 0, 0))) and
            (row['Time'].time() < time(23, 0, 0))):
            long_order = sim.create_order(
                order_type=OrderType.Buy,
                symbol='EURUSD',
                volume=row['position_size'],
                fee=max(0., np.random.normal(0.0001, 0.00003)),
            )
            long_stop_loss = long_order.entry_price - row['stop_distance']
            open_orders[long_order] = long_stop_loss


        if ((row['signal'] < 0) and (row['seconds_to_next_news_event'] > 900) and 
            (row['seconds_since_last_news_event'] < -900) and 
            ((row['Time'].time() >= time(10, 0, 0)) and (row['Time'].time() <= time(17, 0, 0))) and
            (((row['Time'].time() < time(23, 0, 0)) and (row['Time'].weekday() == 4)) or 
            row['Time'].weekday() != 4)): 
            short_order = sim.create_order(
                order_type=OrderType.Sell,
                symbol='EURUSD',
                volume=row['position_size'],
                fee=max(0., np.random.normal(0.0001, 0.00003)),
            )
            short_stop_loss = short_order.entry_price + row['stop_distance']
            open_orders[short_order] = short_stop_loss
    state = sim.get_state()
    if len(state['orders']) > 0:
        reward = state['orders']['Profit'].sum()
    else:
        reward = 0

    print(
        f"balance: {state['balance']}, profit: {reward}, total orders: {len(state['orders'])}"
    )

    reward *= -1 
    if (len(state['orders']) < total_wks):
        reward += float('inf')


    return {'loss': reward, 'status': STATUS_OK, 'eval_time': datetime.now(), 'parameters': params} 
    


In [201]:
from hyperopt import hp

search_space = {
    'bb_period': hp.quniform('bb_period', 10, 50, 1),
    'bb_std': hp.uniform('bb_std', 1.5, 3.0),
    'keltner_period': hp.quniform('keltner_period', 10, 50, 1),
    'keltner_atr_multiplier': hp.uniform('keltner_atr_multiplier', 1.0, 2.5),
    'rsi_period': hp.quniform('rsi_period', 7, 21, 1),
    'stoch_period': hp.quniform('stoch_period', 7, 21, 1),
    'roc_period': hp.quniform('roc_period', 5, 20, 1)
}


In [ ]:
trials = Trials()
best = fmin(fn=objective,
            space=search_space,
            algo=tpe.suggest,
            max_evals=100_000, # Number of evaluations of the objective function
            trials=trials,
            trials_save_file=f'{path_to_trading_folder}Indicator_backtesting/Indicator_search/trials_SDMomentum_{datetime.now().strftime("%Y%m%d_%H%M%S")}.pkl')

print("Best parameters:", best)

In [176]:
positions = state['orders']
positions.loc[:, "Week_Nbr"] = positions['Entry Time'].dt.year.astype(str) + positions['Entry Time'].dt.isocalendar().week.astype(str)
len(positions.Week_Nbr.unique())

382

In [ ]:
positions = state['orders']
positions.loc[:, "Week_Nbr"] = positions['Entry Time'].dt.year.astype(str) + positions['Entry Time'].dt.isocalendar().week.astype(str)
# get the average number of positions per week by getting the distinct count of Week_Nbr column
avg_positions_per_week = positions.groupby('Week_Nbr').size().mean()
print(avg_positions_per_week)
positions